In [2]:
# Justice Chowdary High School
import pyspark.sql.functions as F
from pyspark.sql.window import Window
import pandas as pd

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
            .config("spark.driver.memory", "2g") \
            .appName("justice_chowdary_highschool").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/22 15:08:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
marks_path = "../data/jchs_marks_data.json"
weights_path = "../data/jchs_weightage.csv"

In [5]:
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, IntegerType, DateType



In [8]:
weight_schema = StructType(
    [
    StructField("exam_name", StringType(), True),
    StructField("weightage", IntegerType(), True)]
)

In [16]:
subject_schema = StructType(
    [StructField("Telugu", IntegerType(), True),
    StructField("Hindi", IntegerType(), True),
    StructField("English", IntegerType(), True),
    StructField("Maths", IntegerType(), True),
    StructField("Science", IntegerType(), True),
    StructField("Social", IntegerType(), True)]
)
exam_subject_schema = StructType(
    [StructField("exam", StringType(), True),
    StructField("subjects", subject_schema, True)]
)
student_schema = StructType(
    [StructField("name", StringType(), True),
    StructField("marks", ArrayType(exam_subject_schema), True)]
)

marks_schema = StructType(
    [StructField("school", StringType(), True),
    StructField("academic_year", StringType(), True),
    StructField("students", ArrayType(student_schema), True)]
)


In [17]:
def load_weights(weights_path):
    df_weights = spark.read\
                .format("csv")\
                .option("header",True)\
                .schema(weight_schema)\
                .option("mode","PERMISSIVE")\
                .load(weights_path)
    return df_weights
    
def load_marks(marks_path):
    df_marks = spark.read\
                .format("json")\
                .option("multiline",True)\
                .schema(marks_schema)\
                .load(marks_path)
    return df_marks

In [12]:
def transform_marks(df_marks):
    df2 = df_marks.withColumn("student_info", F.explode(F.col("students")))\
            .withColumn("student_name",F.col("student_info.name"))\
            .withColumn("marks",F.col("student_info.marks"))\
            .drop("students","student_info")

    df5 = df2.withColumn("marks_exp",F.explode("marks"))\
                .withColumn("exam_name",F.col("marks_exp.exam"))\
                .withColumn("subjects",F.col("marks_exp.subjects"))\
                .select('academic_year', 'school', 'student_name', 
                   'exam_name', 'subjects.English',
                   'subjects.Hindi','subjects.Telugu',
                   'subjects.Maths','subjects.Science',
                    'subjects.Social')
                


    df_totals = df5.withColumn("sub_total",F.coalesce(df5.English,F.lit(0))
                         + F.coalesce(df5.Hindi,F.lit(0)) + 
                         F.coalesce(df5.Telugu,F.lit(0)) +
                         F.coalesce(df5.Maths,F.lit(0)) +
                         F.coalesce(df5.Science,F.lit(0)) +
                         F.coalesce(df5.Social,F.lit(0))).\
                        select('academic_year', 'school', 'student_name', 
               'exam_name','sub_total')

    return df_totals

In [13]:
def join_marks_weights(df_totals,df_weights):
    df_joined = df_totals.join(df_weights, "exam_name","left")
    return df_joined

In [14]:
def rank_calculation(df_joined):
    df_weighted = df_joined.withColumn("weighted_score" , F.col("sub_total") * F.col("weightage") / 100 )\
                            .groupBy("student_name").agg(F.round(F.sum("weighted_score"),2).alias("final_score"))
    
    window1 = Window.orderBy(F.col("final_score").desc())
    
    df_final = df_weighted.withColumn("rank",F.row_number().over(window1))

    return df_final

In [18]:
df_marks = load_marks(marks_path)
df_weights = load_weights(weights_path)

df_totals = transform_marks(df_marks)

df_joined = join_marks_weights(df_totals,df_weights)

df_final = rank_calculation(df_joined)

In [19]:
df_final.show(10)

26/08/22 15:11:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/22 15:11:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/22 15:11:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/22 15:11:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/22 15:11:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/22 15:11:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/22 1

+------------+-----------+----+
|student_name|final_score|rank|
+------------+-----------+----+
|        Teja|      536.7|   1|
|         Sai|      519.6|   2|
|       Bhanu|      516.2|   3|
|      Ramesh|      482.6|   4|
|     Bhargav|      472.6|   5|
|     Pradeep|      458.2|   6|
|      Suresh|      442.2|   7|
|        Siva|     429.85|   8|
|      Ganesh|      412.6|   9|
+------------+-----------+----+



In [77]:
df_final.toPandas().to_csv("../data/final_ranks3.csv",index = False)